In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import yaml
import os
from catboost import Pool
from catboost import CatBoostClassifier

from pathlib import Path

from sklearn.model_selection import train_test_split
from src.feature_generation.make_features import make_features_and_save

config_path = 'configs/features_config_30d.yaml'
with open(config_path, 'r') as config_file:
        config = yaml.safe_load(config_file)

In [ ]:
countries_train = ["Dem_Rep_Korea",
                       "Russian_Federation",
                       "Finland",
                       "Norway",
                       "Sweden",
                       # "Denmark",
                       "Lithuania",
                       "Latvia",
                       "Estonia",
                       "Poland",
                       "Czech_Republic",
                       # "Germany",
                       "Hungary",
                       "Slovakia",
                       "Belarus",
                       "Ukraine",
                       "Moldova",
                       "Romania",
                       "Bulgaria",
                       "Albania",
                       "Montenegro",
                       "Macedonia_Former_Yugoslav_Republic_of",
                       "Kosovo",
                       "Serbia",
                       "Croatia",
                       "Bosnia_and_Herzegovina",
                       "Slovenia",
                       "Greece",
                       "Turkey",
                       "Georgia",
                       "Azerbaijan",
                        "Armenia",
                       "Kazakhstan",
                       "Kyrgyzstan",
                       "Tajikistan",
                       "Mongolia",
                       # "China",
                       # "Japan",
                       # "Republic_of_Korea"
                       ]

countries_df_list = []


if not os.path.exists('data/saved_features_boost/train_test_features_30d_all.parquet'):
    for country in countries_train:
        df_country = pd.read_parquet(f'data/saved_features_boost/train_test_features_30d_{country}.parquet')
        if country == "Ukraine":
            # Remove data after 2022-02-24 because of the war
            df_country = df_country[df_country['datetime'] < '2022-02-24']
        countries_df_list.append(df_country)

    df = pd.concat(countries_df_list)
    df = df.sort_values('datetime')
    #save final_df
    df.to_parquet('data/saved_features_boost/train_test_features_30d_all.parquet')
else:
    df = pd.read_parquet('data/saved_features_boost/train_test_features_30d_all.parquet')



with open(config['selected_feature_columns_path'], 'r') as f:
    selected_features = [line.strip() for line in f.readlines()]

#Check if all selected columns are in the dataframe
# assert all(col in df.columns for col in selected_features)


In [4]:
target_col = 'count'
df['population'] = df['population'].fillna(0)
df['population'] = df['population'].astype(int)

cat_features = config['cat_features']
numerical_cat_features = config['numerical_cat_features']

for col in cat_features:
    if col in numerical_cat_features:
        df[col] = df[col].astype(int)
    else:
        df[col] = df[col].astype(str)

X = df.drop(columns=[target_col])
y = df[target_col]
#X  = X[selected_features]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, shuffle=False)

print(f"Train start year: {X_train['datetime'].min().year}")
print(f"Test start year: {X_test['datetime'].min().year}")

Train start year: 2000
Test start year: 2021


In [5]:
y_train_binary = (y_train >0).astype(int)
y_test_binary = (y_test > 0).astype(int)

In [ ]:
MODEL_PATH = "./models/catboost_fire_model_30d.cbm"
OUTPUT_DIR = "./outputs"

In [7]:
model = CatBoostClassifier()
model.load_model(MODEL_PATH)


y_pred_binary = model.predict(X_test)

year=2025
out_dir=OUTPUT_DIR

There are invalid params and some of them will be ignored.
Parameter {"feature_weights":{"ecoregion_name":0.3000000119,"lat_rounded":0.3000000119,"lon_rounded":0.3000000119}} is ignored, because it cannot be parsed.
Parameter {"class_weights":[1,2]} is ignored, because it cannot be parsed.
TBB Warning: The number of workers is currently limited to 5. The request for 63 workers is ignored. Further requests for more workers will be silently ignored until the limit changes.



In [8]:
assert len(X_test) == len(y_test_binary) == len(y_pred_binary)

year_mask = np.array(pd.to_datetime(X_test['datetime']).dt.year == year)

lat = X_test['lat_rounded'][year_mask]
lon = X_test['lon_rounded'][year_mask]
pred_count = y_pred_binary[year_mask]
actual_count = y_test_binary[year_mask]


regions = {
    'Eastern Europe': {'lon_range': (20, 50), 'lat_range': (40, 60)},
    'Scandinavia': {'lon_range': (5, 30), 'lat_range': (55, 75)},
    'Siberia West': {'lon_range': (60, 100), 'lat_range': (50, 70)},
    'Siberia East': {'lon_range': (100, 140), 'lat_range': (50, 70)},
    'Far East Russia': {'lon_range': (140, 180), 'lat_range': (50, 70)},
    'Central Asia': {'lon_range': (50, 80), 'lat_range': (35, 50)},
}

try:
    world = gpd.read_file('data/countries')
except:
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))


out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)


for name, region_info in regions.items():
    lon_range = region_info['lon_range']
    lat_range = region_info['lat_range']
    
    region_mask = (
        (lon >= lon_range[0]) & (lon <= lon_range[1]) &
        (lat >= lat_range[0]) & (lat <= lat_range[1])
    )
    
    if not region_mask.any():
        print(f"No data for region {name}, skipping.")
        continue
    
    lon_region = lon[region_mask]
    lat_region = lat[region_mask]
    actual_region = actual_count[region_mask]
    pred_region = pred_count[region_mask]
    

    fig, ax = plt.subplots(figsize=(10, 8))
    world.plot(ax=ax, color='lightgrey', edgecolor='black')

    if len(lon_region) != len(actual_region) or len(lon_region) != len(pred_region):
        print(f"Array size mismatch for {name}, skipping.")
        plt.close(fig)
        continue

    scatter_pred = ax.scatter(lon_region, 
                              lat_region,
                                c='red', 
                                s=np.clip(pred_region * 23, 0, 400),
                                alpha=0.3, label='Predicted', 
                                edgecolors='none')

    scatter_actual = ax.scatter(lon_region, 
                                lat_region,
                                c='blue', 
                                s=np.clip(actual_region * 12, 0, 400),
                                alpha=0.5, label='Actual',
                                edgecolors='none')
    
    ax.set_xlim(lon_range[0], lon_range[1])
    ax.set_ylim(lat_range[0], lat_range[1])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'{name} ({lon_range[0]}–{lon_range[1]}°E, '
                f'{lat_range[0]}–{lat_range[1]}°N), Year {year}')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save plot
    out_path = out_dir / f"hotspots_{name.replace(' ', '_').lower()}_{year}.png"
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved {name} map to {out_path}")


# fig, ax = plt.subplots(figsize=(15, 10))
# world.plot(ax=ax, color='lightgrey', edgecolor='black')

# scatter_pred_all = ax.scatter(lon, 
#                               lat,
#                                 c='red', 
#                                 s=np.clip(pred_count * 23, 0, 300),
#                                 alpha=0.3, label='Predicted', 
#                                 edgecolors='none')

# scatter_actual_all = ax.scatter(lon, 
#                                 lat,
#                                 c='blue', 
#                                 s=np.clip(actual_count * 12, 0, 300),
#                                 alpha=0.5, label='Actual',
#                                 edgecolors='none')

# ax.set_xlim(20, 170)
# ax.set_ylim(40, 90)
# ax.set_xlabel('Longitude')
# ax.set_ylabel('Latitude')
# ax.set_title(f'Actual vs Predicted Fire Hotspots ({year})')
# ax.legend(loc='upper right')
# ax.grid(True, alpha=0.3)
# plt.tight_layout()

# overall_path = out_dir / f"hotspots_{year}.png"
# fig.savefig(overall_path, dpi=200, bbox_inches='tight')
# plt.close(fig)
# print(f"Saved overall map to {overall_path}")

Saved Eastern Europe map to output/hotspots_eastern_europe_2025.png
Saved Scandinavia map to output/hotspots_scandinavia_2025.png
Saved Siberia West map to output/hotspots_siberia_west_2025.png
Saved Siberia East map to output/hotspots_siberia_east_2025.png
Saved Far East Russia map to output/hotspots_far_east_russia_2025.png
Saved Central Asia map to output/hotspots_central_asia_2025.png


In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
world.plot(ax=ax, color='lightgrey', edgecolor='black')

pred_sizes = np.clip(23 * pred_count, 0, 300)
actual_sizes = np.clip(12 * actual_count, 0, 300)

scatter_pred = ax.scatter(lon, 
                          lat,
                            c='red', s=pred_sizes, 
                            alpha=0.4, label='Predicted', 
                            edgecolors='none')

scatter_actual = ax.scatter(lon, 
                            lat,
                            c='blue', s=actual_sizes, 
                            alpha=0.6, label='Actual',
                            edgecolors='none')

for i, (name, region_info) in enumerate(regions.items()):
        lon_range = region_info['lon_range']
        lat_range = region_info['lat_range']
        
        rect = plt.Rectangle((lon_range[0], lat_range[0]),
                           lon_range[1] - lon_range[0],
                           lat_range[1] - lat_range[0],
                           fill=False, 
                           linewidth=2, linestyle='--', alpha=0.8)
        
        ax.add_patch(rect)
        
        ax.text(lon_range[0] + (lon_range[1] - lon_range[0])/2,
               lat_range[1] + 1, name,
               ha='center', va='bottom',
               fontsize=9, weight='bold',
               bbox=dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.7))
    
ax.set_xlim(0, 180)
ax.set_ylim(30, 90)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Overview: Actual vs Predicted Fire Hotspots with Region Boundaries ({year})')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()

out_path = out_dir / f"overview_with_regions_{year}.png"
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, dpi=200, bbox_inches='tight')
plt.close(fig)
print(f"Saved overview map with regions to {out_path}")